In [7]:
import pandas as pd

def build_hotonly_summary(
    monthly_path="v260915_hotonly_monthly_total__ADM1_1996-2025_rounded.csv",
    adm1_6mo_path="v260915_hotonly_6mo_total__ADM1_1996-2025_rounded.csv",
    iso_6mo_path="v260915_hotonly_6mo_total__ISO_1996-2025_rounded.csv",
    names_path="/home/emily_zuetell/projects/poreallas/data/adm1.parquet",
    output_path="v3_map.csv",
):
    month_cols = {
        "month 9 mean": "Sep 2026",
        "month 10 mean": "Oct",
        "month 11 mean": "Nov",
        "month 12 mean": "Dec",
        "month 1 mean": "Jan 2027",
        "month 2 mean": "Feb",
    }
    monthly = pd.read_csv(monthly_path, usecols=["GID_0", "GID_1", *month_cols])
    monthly = monthly.rename(columns=month_cols)
 
    adm1_6mo = pd.read_csv(adm1_6mo_path, usecols=["GID_0", "GID_1", "mean"])
    adm1_6mo = adm1_6mo.rename(columns={"mean": "6mo_hotonly_total_mean"})
 
    iso_6mo = pd.read_csv(iso_6mo_path, usecols=["ISO", "mean"])
    iso_6mo = iso_6mo.rename(columns={"ISO": "GID_0", "mean": "country_6mo_hotonly_total"})
 
    names = pd.read_parquet(names_path, columns=["GID_0", "GID_1", "NAME_0", "NAME_1"])
    # Countries with no ADM1 subdivision (e.g. ABW) have GID_1/NAME_1 = NaN in the
    # lookup, but the data files use GID_1 == GID_0 for those rows. Fill so they merge.
    no_adm1 = names["GID_1"].isna() | (names["GID_1"].str.strip() == "")
    names.loc[no_adm1, "GID_1"] = names.loc[no_adm1, "GID_0"]
    names.loc[no_adm1, "NAME_1"] = names.loc[no_adm1, "NAME_0"]
 
    df = (
        monthly
        .merge(adm1_6mo, on=["GID_0", "GID_1"], how="left")
        .merge(iso_6mo, on="GID_0", how="left")
        .merge(names, on=["GID_0", "GID_1"], how="left")
    )
 
    df["name"] = df["NAME_1"].astype(str) + ", " + df["NAME_0"].astype(str)
 
    df = df[[
        "GID_0", "GID_1", "NAME_0", "NAME_1",
        "Sep 2026", "Oct", "Nov", "Dec", "Jan 2027", "Feb",
        "6mo_hotonly_total_mean", "country_6mo_hotonly_total", "name",
    ]]
    df["country_6mo_hotonly_total"] = df["country_6mo_hotonly_total"].astype("Int64")
 
    df.to_csv(output_path, index=False)
    return df

def to_long_form(df, output_path="v3_tooltip.csv"):
    month_cols = ["Sep 2026", "Oct", "Nov", "Dec", "Jan 2027", "Feb"]
    long_df = df.melt(
        id_vars=["GID_0", "GID_1", "NAME_0", "NAME_1"],
        value_vars=month_cols,
        var_name="month",
        value_name="hot deaths",
    )
    long_df.to_csv(output_path, index=False)
    return long_df

In [8]:
df = build_hotonly_summary()
long_df = to_long_form(df)

In [9]:
long_df

,GID_0,GID_1,NAME_0,NAME_1,month,hot deaths
0,ABW,ABW,Aruba,Aruba,Sep 2026,0
1,AFG,AFG.10_1,Afghanistan,Ghor,Sep 2026,0
2,AFG,AFG.11_1,Afghanistan,Hilmand,Sep 2026,12
3,AFG,AFG.12_1,Afghanistan,Hirat,Sep 2026,-8
4,AFG,AFG.13_1,Afghanistan,Jawzjan,Sep 2026,-3
...,...,...,...,...,...,...
22093,ZWE,ZWE.5_1,Zimbabwe,Mashonaland East,Feb,24
22094,ZWE,ZWE.6_1,Zimbabwe,Mashonaland West,Feb,42
22095,ZWE,ZWE.7_1,Zimbabwe,Masvingo,Feb,36
22096,ZWE,ZWE.8_1,Zimbabwe,Matabeleland North,Feb,42
